In [1]:
from google.colab import files

uploaded = files.upload()

Saving Cars Datasets 2025.csv to Cars Datasets 2025.csv


In [5]:
import pandas as pd

cars = pd.read_csv('Cars Datasets 2025.csv', encoding='latin1')


cars.head()

,Company Names,Cars Names,Engines,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Fuel Types,Seats,Torque
0,FERRARI,SF90 STRADALE,V8,3990 cc,963 hp,340 km/h,2.5 sec,"$1,100,000",plug in hyrbrid,2,800 Nm
1,ROLLS ROYCE,PHANTOM,V12,6749 cc,563 hp,250 km/h,5.3 sec,"$460,000",Petrol,5,900 Nm
2,Ford,KA+,1.2L Petrol,"1,200 cc",70-85 hp,165 km/h,10.5 sec,"$12,000-$15,000",Petrol,5,100 - 140 Nm
3,MERCEDES,GT 63 S,V8,"3,982 cc",630 hp,250 km/h,3.2 sec,"$161,000",Petrol,4,900 Nm
4,AUDI,AUDI R8 Gt,V10,"5,204 cc",602 hp,320 km/h,3.6 sec,"$253,290",Petrol,2,560 Nm


In [6]:
cars.head().shape

(5, 11)

In [9]:
cars.isnull().sum()

,0
Company Names,0
Cars Names,0
Engines,0
CC/Battery Capacity,3
HorsePower,0
Total Speed,0
Performance(0 - 100 )KM/H,6
Cars Prices,0
Fuel Types,0
Seats,0


In [13]:
cars = cars[['Company Names','Cars Names','Cars Prices','Fuel Types','Seats']]

In [16]:
cars.head()

,Company Names,Cars Names,Cars Prices,Fuel Types,Seats
0,FERRARI,SF90 STRADALE,"$1,100,000",plug in hyrbrid,2
1,ROLLS ROYCE,PHANTOM,"$460,000",Petrol,5
2,Ford,KA+,"$12,000-$15,000",Petrol,5
3,MERCEDES,GT 63 S,"$161,000",Petrol,4
4,AUDI,AUDI R8 Gt,"$253,290",Petrol,2


In [47]:
import numpy as np
import pandas as pd

# 1. Standardize Company Names to uppercase
cars["Company Names"] = cars["Company Names"].str.upper()

# 2. Fix the spelling mistake in Fuel Types
cars["Fuel Types"] = cars["Fuel Types"].str.replace(
    "hyrbrid", "hybrid", case=False
)


def clean_price(price_str):
    try:
        price_str = str(price_str).strip()
        price_str = price_str.replace('$', '').replace(',', '')

        if '/' in price_str:
            low, high = price_str.split('/')
            return (float(low) + float(high)) / 2

        return float(price_str)

    except:
        return np.nan
cars["Cars Prices"] = cars["Cars Prices"].apply(clean_price)
print(cars)


     Company Names         Cars Names  Cars Prices               Fuel Types  \
0          FERRARI      SF90 STRADALE    1100000.0           plug in hybrid   
1      ROLLS ROYCE            PHANTOM     460000.0                   Petrol   
2             FORD                KA+          NaN                   Petrol   
3         MERCEDES            GT 63 S     161000.0                   Petrol   
4             AUDI         AUDI R8 Gt     253290.0                   Petrol   
...            ...                ...          ...                      ...   
1213        TOYOTA       Crown Signia          NaN  Hybrid (Gas + Electric)   
1214        TOYOTA  4Runner (6th Gen)      50000.0                   Hybrid   
1215        TOYOTA      Corolla Cross          NaN             Gas / Hybrid   
1216        TOYOTA             C-HR+           NaN                   Hybrid   
1217        TOYOTA     RAV4 (6th Gen)          NaN         Hybrid / Plug-in   

      Seats                          Features  
0  

In [28]:
cars["Company Names"] = cars["Company Names"].fillna("").astype(str)
cars["Fuel Types"] = cars["Fuel Types"].fillna("").astype(str)
cars["Seats"] = cars["Seats"].fillna(0).astype(str)

cars["Features"] = (
    cars["Company Names"] + " " +
    cars["Fuel Types"] + " " +
    cars["Seats"]
)

In [30]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()

feature_matrix = cv.fit_transform(cars["Features"])
print(feature_matrix.shape)

(1218, 51)


In [31]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(feature_matrix)

print(similarity.shape)

(1218, 1218)


In [43]:
def recommend_car(budget, fuel, seats):

    result = cars[
        (cars["Cars Prices"] <= budget) &
        (cars["Fuel Types"].str.contains(fuel, case=False, na=False)) &
        (cars["Seats"] >= seats)
    ]

    return result.sort_values(
        by="Cars Prices",
        ascending=False
    )[[
        "Company Names",
        "Cars Names",
        "Cars Prices",
        "Fuel Types",
        "Seats"
    ]]

In [40]:
print(cars["Seats"].dtype)
print(cars["Seats"].head())

object
0    2
1    5
2    5
3    4
4    2
Name: Seats, dtype: object


In [41]:
cars["Seats"] = pd.to_numeric(cars["Seats"], errors="coerce")
cars["Seats"] = cars["Seats"].fillna(0).astype(int)

In [48]:
budget = float(input("Enter your budget: "))
fuel = input("Fuel Type: ")
company = input("Preferred Company (or press Enter): ")
seats = int(input("Minimum Seats: "))

if company:
    result = result[
        result["Company Names"].str.contains(
            company,
            case=False,
            na=False
        )
    ]

print(
    recommend_car(
        budget,
        fuel,
        seats
    )
)


Enter your budget: 100000000
Fuel Type: petrol
Preferred Company (or press Enter): 
Minimum Seats: 2
    Company Names        Cars Names  Cars Prices     Fuel Types  Seats
887       BUGATTI  La Voiture Noire   18000000.0         Petrol      2
886       BUGATTI        Centodieci    9000000.0         Petrol      2
885       BUGATTI              Divo    5800000.0         Petrol      2
889       BUGATTI           Mistral    5000000.0         Petrol      2
888       BUGATTI            Bolide    4500000.0         Petrol      2
..            ...               ...          ...            ...    ...
643   TATA MOTORS     Indigo Marina       8300.0  Diesel/Petrol      5
628   TATA MOTORS             Tiago       8200.0         Petrol      5
646   TATA MOTORS        Indigo GLX       7200.0         Petrol      5
637   TATA MOTORS    Indica V2 Xeta       5000.0         Petrol      5
635   TATA MOTORS         Nano GenX       4000.0         Petrol      4

[776 rows x 5 columns]
